# Phase 1: Data Models

This notebook demonstrates all Pydantic data models from `researcher_ai.models`.
It validates that the package is installed correctly and shows JSON schemas for each model.

**Purpose:** Establish the data contract between all components of `researcher-ai`.

In [ ]:
import json
from IPython.display import JSON

# Import all models
from researcher_ai.models import (
    Paper, PaperSource, PaperType, Section, Reference, SupplementaryItem,
    Figure, SubFigure, PlotCategory, PlotType, PlotLayer,
    Axis, AxisScale, ColorMapping, ColormapType,
    ErrorBarType, StatisticalAnnotation, PanelLayout, RenderingSpec,
    Method, Assay, AnalysisStep, AssayGraph, AssayDependency,
    Dataset, GEODataset, SRADataset, ProteomicsDataset, DataSource,
    Software, Command, Environment, LicenseType,
    Pipeline, PipelineConfig, PipelineStep, PipelineBackend,
)

print("All models imported successfully.")

## Paper model

In [ ]:
# Show Paper JSON schema
print(json.dumps(Paper.model_json_schema(), indent=2))

In [ ]:
# Construct a minimal Paper object
paper = Paper(
    title="Robust transcriptome-wide discovery of RNA-binding protein binding sites with enhanced CLIP (eCLIP)",
    authors=["Van Nostrand, E.L.", "Pratt, G.A.", "Shishkin, A.A.", "Yeo, G.W."],
    doi="10.1038/nmeth.3810",
    pmid="26971820",
    source=PaperSource.PMID,
    source_path="26971820",
    paper_type=PaperType.EXPERIMENTAL,
)

print(paper.model_dump_json(indent=2))

## Figure model

In [ ]:
# Construct a Figure with a volcano plot subfigure (common in Yeo Lab papers)
volcano_sf = SubFigure(
    label="a",
    description="Volcano plot of differentially expressed genes (KD vs. control)",
    plot_category=PlotCategory.GENOMIC,
    plot_type=PlotType.VOLCANO,
    x_axis=Axis(label="log2 Fold Change", scale=AxisScale.LINEAR),
    y_axis=Axis(label="-log10(p-value)", scale=AxisScale.LINEAR, is_inverted=False),
    color_mapping=ColorMapping(
        variable="significance",
        colormap_type=ColormapType.QUALITATIVE,
        colormap_name="Okabe-Ito",
    ),
    error_bars=ErrorBarType.NONE,
    sample_size="n=3 biological replicates",
    statistical_annotations=StatisticalAnnotation(
        test_name="DESeq2 Wald test",
        multiple_testing_correction="BH",
    ),
)

fig = Figure(
    figure_id="Figure 2",
    title="Transcriptome-wide changes upon RBP knockdown",
    caption="Figure 2. (a) Volcano plot showing...",
    purpose="Show which genes are differentially expressed upon RBP knockdown",
    subfigures=[volcano_sf],
    layout=PanelLayout(n_rows=1, n_cols=1),
)

print(fig.model_dump_json(indent=2))

## Method + AssayGraph model

In [ ]:
# Demonstrate DAG-structured assay dependencies
# Example: eCLIP paper where CLIP-seq signal normalization depends on matched input

eclip = Assay(
    name="eCLIP",
    description="Enhanced CLIP-seq for RBP binding sites",
    data_type="sequencing",
    raw_data_source="GEO: GSE78509",
    steps=[
        AnalysisStep(
            step_number=1,
            description="Align reads with STAR",
            input_data="demultiplexed FASTQ",
            output_data="BAM",
            software="STAR",
            software_version="2.7.10a",
        ),
        AnalysisStep(
            step_number=2,
            description="Remove PCR duplicates",
            input_data="BAM",
            output_data="deduplicated BAM",
            software="umi_tools",
        ),
    ],
)

matched_input = Assay(
    name="SMInput",
    description="Size-matched input control for eCLIP normalization",
    data_type="sequencing",
)

peak_calling = Assay(
    name="Peak Calling",
    description="Identify enriched binding sites vs. SMInput",
    data_type="analysis",
)

assay_graph = AssayGraph(
    assays=[eclip, matched_input, peak_calling],
    dependencies=[
        AssayDependency(
            upstream_assay="eCLIP",
            downstream_assay="Peak Calling",
            dependency_type="normalization_reference",
            description="eCLIP BAMs used as IP input to CLIPper",
        ),
        AssayDependency(
            upstream_assay="SMInput",
            downstream_assay="Peak Calling",
            dependency_type="normalization_reference",
            description="SMInput BAMs used as control for peak enrichment",
        ),
    ],
)

method = Method(
    paper_doi="10.1038/nmeth.3810",
    assay_graph=assay_graph,
    data_availability="GEO: GSE78509",
)

print(f"Assays: {[a.name for a in method.assays]}")
print(f"Upstream of 'Peak Calling': {method.assay_graph.upstream_of('Peak Calling')}")
print()
print(method.model_dump_json(indent=2))

## Dataset models

In [ ]:
geo = GEODataset(
    accession="GSE78509",
    title="eCLIP of RBPs in K562 and HepG2",
    organism="Homo sapiens",
    experiment_type="eCLIP",
    series_type="SuperSeries",
    child_series=["GSE78506", "GSE78507"],
)

print(geo.model_dump_json(indent=2))

## Pipeline model with DAG step ordering

In [ ]:
steps = [
    PipelineStep(
        step_id="download",
        name="Download FASTQ",
        description="Download raw eCLIP reads from SRA",
        software="fasterq-dump",
        command="fasterq-dump {srr} -O {outdir}",
    ),
    PipelineStep(
        step_id="align",
        name="Align reads",
        description="Align to hg38 with STAR",
        software="STAR",
        command="STAR --runMode alignReads --readFilesIn {fastq}",
        depends_on=["download"],
        threads=8,
        memory_gb=32,
    ),
    PipelineStep(
        step_id="dedup",
        name="Remove PCR duplicates",
        description="UMI-based deduplication",
        software="umi_tools",
        command="umi_tools dedup -I {bam} -S {out_bam}",
        depends_on=["align"],
    ),
    PipelineStep(
        step_id="peaks",
        name="Call peaks",
        description="CLIPper peak calling vs. SMInput",
        software="CLIPper",
        command="clipper -b {ip_bam} --outfile {peaks_bed}",
        depends_on=["dedup"],
    ),
]

config = PipelineConfig(
    name="eclip_pipeline",
    description="Reproduce eCLIP analysis from Van Nostrand et al. 2016",
    backend=PipelineBackend.SNAKEMAKE,
    steps=steps,
    datasets=["GSE78509"],
    figure_targets=["Figure 1", "Figure 2"],
)

print("Execution order:", config.execution_order())

pipeline = Pipeline(config=config)
print(pipeline.model_dump_json(indent=2))

## JSON schema dump (all models)

These schemas define the data contract for all downstream components.

In [ ]:
models = {
    "Paper": Paper,
    "Figure": Figure,
    "SubFigure": SubFigure,
    "Method": Method,
    "Dataset": Dataset,
    "Software": Software,
    "Pipeline": Pipeline,
    "PipelineConfig": PipelineConfig,
    "PipelineStep": PipelineStep,
}

for name, model_cls in models.items():
    schema = model_cls.model_json_schema()
    fields = list(schema.get("properties", {}).keys())
    print(f"{name}: {len(fields)} fields → {fields}")